<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/iaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inter-Annotator Agreement (IAA) — Krippendorff's α via BERTScore

Pipeline:
1. Pull `human_insights` rows from Supabase
2. Build (dashboard_id, chart_id, level) units
3. Compute BERTScore-based pairwise distances for all 3 annotator pairs
4. Compute Krippendorff's α overall, per level, and per dashboard
5. Export results to Excel

## Dependencies

In [50]:
!pip install -q bert-score supabase pandas numpy openpyxl

## STEP 1: Pull data from Supabase

In [51]:
import pandas as pd
import numpy as np
import re
from bert_score import score as bert_score
from supabase import create_client
from itertools import combinations
from google.colab import userdata

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
response = supabase.table("human_insights").select("*").execute()
df_raw = pd.DataFrame(response.data)

print("irr_flag unique values:", df_raw["irr_flag"].unique())
irr_df = df_raw[df_raw["irr_flag"] == True].reset_index(drop=True)
print(f"IRR rows: {len(irr_df)}")

irr_flag unique values: [False  True]
IRR rows: 10


## STEP 2: Parsing step

In [52]:
NA_PATTERN = re.compile(r"^(not applicable|n/a)$", re.IGNORECASE)

def normalize_value(val):
    """Post-parse normalization: catch any NA variants that slipped through."""
    if val is None:
        return None
    cleaned = str(val).strip()
    if cleaned == "" or NA_PATTERN.match(cleaned):
        return None
    return cleaned

def parse_charts(text):
    charts = []
    if not text or (isinstance(text, float) and np.isnan(text)):
        return charts

    blocks = re.split(r"(?=Chart\s+\d+\s*[:.])", text.strip())
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        header = lines[0].strip()
        match = re.match(r"Chart\s+(\d+)\s*[:.]\s*(.*)", header)
        if not match:
            continue
        chart_id = int(match.group(1))
        title    = match.group(2).strip()

        L2 = L3 = L4 = None
        for line in lines[1:]:
            line = line.strip()
            if line.startswith("L2:"):
                L2 = normalize_value(line[3:].strip())
            elif line.startswith("L3:"):
                L3 = normalize_value(line[3:].strip())
            elif line.startswith("L4:"):
                L4 = normalize_value(line[3:].strip())

        charts.append({
            "chart_id": chart_id,
            "title":    title,
            "L2":       L2,
            "L3":       L3,
            "L4":       L4,
        })
    return charts

In [53]:
ANNOTATOR_COLS = ["insight_part_1", "insight_part_2", "insight_part_3"]

records = []
for _, row in irr_df.iterrows():
    row_id      = row["id"]
    metadata_id = row["metadata_id"]

    # Parse each annotator's blob
    parsed = {col: {c["chart_id"]: c for c in parse_charts(row[col])}
              for col in ANNOTATOR_COLS}

    # All chart_ids seen across any annotator
    all_chart_ids = sorted(
        set().union(*[set(p.keys()) for p in parsed.values()])
    )

    for chart_id in all_chart_ids:
        # Get title from whichever annotator has this chart
        title = next(
            (parsed[col][chart_id]["title"]
             for col in ANNOTATOR_COLS
             if chart_id in parsed[col]),
            ""
        )
        for level in ["L2", "L3", "L4"]:
            records.append({
                "row_id":      row_id,
                "metadata_id": metadata_id,
                "chart_id":    chart_id,
                "title":       title,
                "level":       level,
                "unit_id":     f"{row_id}__chart{chart_id}__{level}",
                "annotator_1": parsed["insight_part_1"].get(chart_id, {}).get(level),
                "annotator_2": parsed["insight_part_2"].get(chart_id, {}).get(level),
                "annotator_3": parsed["insight_part_3"].get(chart_id, {}).get(level),
            })

long_df = pd.DataFrame(records)
print(f"\nTotal units: {len(long_df)}")
print(long_df.head(12).to_string())


Total units: 147
                                  row_id                           metadata_id  chart_id                        title level                                           unit_id                                                                                                                                                                                                                                                                                              annotator_1                                                                                                                                                                                                                                                                                                               annotator_2                                                                                                                                                                                                      

In [54]:
print("\nNOT APPLICABLE counts per level per annotator:")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}: "
          f"ann1={sub['annotator_1'].isna().sum()} | "
          f"ann2={sub['annotator_2'].isna().sum()} | "
          f"ann3={sub['annotator_3'].isna().sum()} | "
          f"total_units={len(sub)}")


NOT APPLICABLE counts per level per annotator:
  L2: ann1=0 | ann2=0 | ann3=4 | total_units=49
  L3: ann1=24 | ann2=3 | ann3=2 | total_units=49
  L4: ann1=0 | ann2=0 | ann3=1 | total_units=49


In [55]:
# Check chart count per dashboard per annotator
print("=== Chart count per dashboard per annotator ===\n")

for row_id in sorted(long_df["row_id"].unique()):
    sub = long_df[long_df["row_id"] == row_id]

    # Get unique charts seen per annotator
    ann1_charts = sub[sub["annotator_1"].notna()]["chart_id"].unique()
    ann2_charts = sub[sub["annotator_2"].notna()]["chart_id"].unique()
    ann3_charts = sub[sub["annotator_3"].notna()]["chart_id"].unique()

    total_charts = sub["chart_id"].nunique()
    metadata_id = sub["metadata_id"].iloc[0]

    print(f"Dashboard: {row_id[:8]}… (metadata: {metadata_id[:8]}…)")
    print(f"  Total charts parsed: {total_charts}")
    print(f"  ann1 charts with content: {sorted(ann1_charts)} ({len(ann1_charts)})")
    print(f"  ann2 charts with content: {sorted(ann2_charts)} ({len(ann2_charts)})")
    print(f"  ann3 charts with content: {sorted(ann3_charts)} ({len(ann3_charts)})")

    # Flag mismatches
    all_charts = set(sub["chart_id"].unique())
    for ann_label, ann_charts in [("ann1", ann1_charts), ("ann2", ann2_charts), ("ann3", ann3_charts)]:
        missing = all_charts - set(ann_charts)
        if missing:
            print(f"  ⚠️  {ann_label} missing charts: {sorted(missing)}")
    print()

=== Chart count per dashboard per annotator ===

Dashboard: 08006d53… (metadata: 27052e58…)
  Total charts parsed: 4
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)

Dashboard: 1120a289… (metadata: 4f4b551b…)
  Total charts parsed: 5
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(5)] (4)
  ⚠️  ann3 missing charts: [np.int64(4)]

Dashboard: 2e5b2881… (metadata: 932c11c0…)
  Total charts parsed: 6
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)] (6)
  ann2 charts with content: [np.int64(1), np.int64(2), np

## STEP 3: BERTScore-based distance function

`distance(a, b) = 1 - BERTScore_F1(a, b)`  
`NOT APPLICABLE` entries are treated as missing (`np.nan`).

In [56]:
import torch
import numpy as np
from bert_score import score as bert_score
from itertools import combinations

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_TYPE = "roberta-large"
print(f"Using device: {DEVICE}, model: {MODEL_TYPE}")

ANNOTATOR_COLS = ["annotator_1", "annotator_2", "annotator_3"]
ANNOTATOR_PAIRS = list(combinations(ANNOTATOR_COLS, 2))

def is_missing(text):
    if text is None:
        return True
    return str(text).strip().lower() in {"not applicable", "n/a", ""}

def bertscore_distance_batch(refs, hyps, model_type=MODEL_TYPE, device=DEVICE):
    """
    Compute BERTScore F1 for parallel lists of refs and hyps.
    Returns a numpy array of distances (1 - F1).
    Missing values return np.nan.
    """
    assert len(refs) == len(hyps)
    distances = np.full(len(refs), np.nan)

    valid_indices = [
        i for i, (r, h) in enumerate(zip(refs, hyps))
        if not is_missing(r) and not is_missing(h)
    ]

    if not valid_indices:
        return distances

    valid_refs = [str(refs[i]) for i in valid_indices]
    valid_hyps = [str(hyps[i]) for i in valid_indices]

    _, _, F1 = bert_score(
        valid_hyps, valid_refs,
        model_type=model_type,
        device=device,
        verbose=False
    )

    for idx, f1_val in zip(valid_indices, F1.numpy()):
        distances[idx] = 1.0 - f1_val

    return distances

# ============================================================
# Compute pairwise distances on long_df
# ============================================================
DIST_COLS = []
for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"dist_{suffix_a}{suffix_b}"
    DIST_COLS.append(pair_key)
    print(f"Computing {pair_key} ({col_a} vs {col_b}) ...")
    long_df[pair_key] = bertscore_distance_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist(),
    )

print("\nSample distances:")
print(long_df[["unit_id", "chart_id", "level"] + DIST_COLS].head(12).to_string(index=False))

Using device: cpu, model: roberta-large
Computing dist_12 (annotator_1 vs annotator_2) ...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

KeyboardInterrupt: 

## STEP 4: Krippendorff's alpha with custom BERTScore distance

$$\alpha = 1 - \frac{D_o}{D_e}$$

- **D_o** = mean observed disagreement (average pairwise distance within units, >=2 valid annotators)
- **D_e** = mean expected disagreement (average over all valid distances in the pool)

In [ ]:
from bert_score import score as bert_score
import numpy as np

def compute_cross_item_distances_batched(sub_df, ann_col, model_type="distilbert-base-uncased", device=DEVICE):
    """
    Batch all cross-unit pairs for one annotator column.
    D_e = distances between annotations from DIFFERENT units.
    Returns list of float distances (1 - BERTScore F1).
    """
    annotations = sub_df[ann_col].tolist()
    n = len(annotations)

    refs_batch = []
    hyps_batch = []

    for i in range(n):
        for j in range(i + 1, n):
            a, b = annotations[i], annotations[j]
            if is_missing(a) or is_missing(b):
                continue
            refs_batch.append(str(a))
            hyps_batch.append(str(b))

    if not refs_batch:
        return []

    _, _, F1 = bert_score(
        hyps_batch, refs_batch,
        model_type=model_type,
        device=device,
        verbose=False
    )
    return (1.0 - F1.numpy()).tolist()


def krippendorff_alpha_bertscore_correct(sub_df, dist_cols=DIST_COLS):
    """
    Corrected Krippendorff's alpha following Braylan et al. (2022):

    D_o = mean within-item pairwise distances
          (already computed as dist_12, dist_13, dist_23 per unit)

    D_e = mean cross-item pairwise distances
          (annotations from DIFFERENT units, per annotator)

    α = 1 - (D_o / D_e)
    """
    # --------------------------------------------------------
    # D_o: observed disagreement — within-unit pairs
    # --------------------------------------------------------
    unit_mean_dist = sub_df[dist_cols].mean(axis=1, skipna=True)
    valid_units    = unit_mean_dist.notna()
    n_valid        = valid_units.sum()

    if n_valid < 2:
        return np.nan, np.nan, np.nan, n_valid

    D_o = unit_mean_dist[valid_units].mean()

    # --------------------------------------------------------
    # D_e: expected disagreement — cross-unit pairs
    # --------------------------------------------------------
    cross_item_distances = []
    for ann_col in ["annotator_1", "annotator_2", "annotator_3"]:
        print(f"  [{ann_col}] computing cross-item distances "
              f"({sub_df[ann_col].notna().sum()} valid annotations)...")
        cross_item_distances.extend(
            compute_cross_item_distances_batched(sub_df, ann_col)
        )

    if not cross_item_distances:
        return np.nan, np.nan, np.nan, n_valid

    D_e = np.mean(cross_item_distances)

    if D_e == 0:
        return np.nan, np.nan, np.nan, n_valid

    alpha = 1.0 - (D_o / D_e)
    return alpha, D_o, D_e, n_valid

## STEP 6: Overall alpha (all dashboards combined)

In [ ]:
print("=" * 60)
print("Krippendorff's α with BERTScore distance (Braylan et al., 2022)")
print("=" * 60)

print("\n[Overall]")
alpha_all, Do_all, De_all, n_all = krippendorff_alpha_bertscore_correct(long_df)
print(f"  α={alpha_all:.4f}  D_o={Do_all:.4f}  D_e={De_all:.4f}  n_valid={n_all}")

## STEP 7: Alpha per insight level (L2 / L3 / L4)

In [ ]:
print("\n[By Semantic Level]")
level_results = []
for level in ["L2", "L3", "L4"]:
    print(f"\n  Level {level}:")
    sub = long_df[long_df["level"] == level]
    alpha, D_o, D_e, n_valid = krippendorff_alpha_bertscore_correct(sub)
    level_results.append({
        "Level":         level,
        "Alpha":         round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "D_o":           round(D_o, 4)   if not np.isnan(D_o)   else "N/A",
        "D_e":           round(D_e, 4)   if not np.isnan(D_e)   else "N/A",
        "N_units_valid": n_valid,
        "N_units_total": len(sub),
    })
    print(f"  → α={alpha:.4f}  D_o={D_o:.4f}  D_e={D_e:.4f}  "
          f"valid={n_valid}/{len(sub)}")

level_df = pd.DataFrame(level_results)

## STEP 8: Alpha per dashboard

In [ ]:

print("\n--- Alpha by Dashboard ---")
dash_results = []
for row_id in sorted(long_df["row_id"].unique()):
    sub = long_df[long_df["row_id"] == row_id]
    alpha, D_o, D_e, n_valid = krippendorff_alpha_bertscore(sub)
    dash_results.append({
        "row_id":        row_id,
        "Alpha":         round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "N_units_valid": n_valid,
        "N_units_total": len(sub),
    })
    print(f"  {row_id[:8]}…: α={alpha:.4f}  valid={n_valid}/{len(sub)}")

dash_df = pd.DataFrame(dash_results)

## STEP 9: Summary printout

In [ ]:
print("\n[Summary Table]")
print(level_df.to_string(index=False))

## STEP 10: Moving on to BERTScore

In [ ]:
F1_COLS = {
    "dist_12": "F1_ann1_ann2",
    "dist_13": "F1_ann1_ann3",
    "dist_23": "F1_ann2_ann3",
}

for dist_col, f1_col in F1_COLS.items():
    long_df[f1_col] = 1.0 - long_df[dist_col]  # NaN stays NaN

F1_VALUE_COLS = list(F1_COLS.values())

# ============================================================
# Mean pairwise F1 per level
# ============================================================
print("--- Mean Pairwise BERTScore F1 by Level ---")
iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    for f1_col in F1_VALUE_COLS:
        mean_f1 = sub[f1_col].mean(skipna=True)
        n_valid  = sub[f1_col].notna().sum()
        row[f1_col]               = round(mean_f1, 4)
        row[f1_col + "_n_valid"]  = n_valid
    row["Mean_F1_overall"] = round(
        sub[F1_VALUE_COLS].values.flatten()[
            ~np.isnan(sub[F1_VALUE_COLS].values.flatten())
        ].mean(), 4
    )
    iaa_results.append(row)
    print(f"  {level}: "
          f"ann1-ann2={row['F1_ann1_ann2']:.4f} (n={row['F1_ann1_ann2_n_valid']})  "
          f"ann1-ann3={row['F1_ann1_ann3']:.4f} (n={row['F1_ann1_ann3_n_valid']})  "
          f"ann2-ann3={row['F1_ann2_ann3']:.4f} (n={row['F1_ann2_ann3_n_valid']})  "
          f"mean={row['Mean_F1_overall']:.4f}")

iaa_df = pd.DataFrame(iaa_results)